In [0]:
data = [
    ("C001", "2024-01-01"),
    ("C001", "2024-01-02"),
    ("C001", "2024-01-04"),
    ("C001", "2024-01-06"),
    ("C002", "2024-01-03"),
    ("C002", "2024-01-05"),
]

df = spark.createDataFrame(data, ["customer_id", "billing_date"])
display(df)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df = df.withColumn('billing_date',col('billing_date').cast('date'))

In [0]:
df_dates = df.groupBy('customer_id').agg(min("billing_date").alias('min_date'),
                                         max("billing_date").alias('max_date'))\
              .withColumn("date",explode(sequence(col("min_date"),col("max_date"))))
display(df_dates)

In [0]:
missing = df_dates.join(df, (df_dates.customer_id == df.customer_id) & (df_dates.date == df.billing_date), how="left_anti").withColumnRenamed('date',"missing_date")
missing.show()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

window_spec = Window.partitionBy('customer_id').orderBy('missing_date')
missing_df = missing.withColumn("prev_date",lag("missing_date").over(window_spec))
missing_df.show()

In [0]:
missing_df=missing_df.withColumn('grp',when(col("prev_date").isNull(),1).when(datediff(col('missing_date'),col('prev_date'))>1,1).otherwise(0))
missing_df.show()

In [0]:
missing_df = missing_df.withColumn('grp_id',sum('grp').over(window_spec)).withColumn('new_grp_id',sum('grp').over(window_spec.rowsBetween(Window.unboundedPreceding,0)))
missing_df.show()

In [0]:
result = missing_df.groupBy("customer_id","new_grp_id").agg(min("missing_date").alias("missing_from"),max("missing_date").alias("missing_to"))
display(result)

### this below code will lead to incorrect result. so we need to calculate groupid, in order for grouping by, so that we'll get correct result as per above code.
# 

In [0]:
missing = missing.groupBy("customer_id").agg(min("missing_date").alias("missing_from"),max("missing_date").alias("missing_to"))
display(missing)